# VERA TERM-RL Training — Language-Table
Phase 2: RL fine-tuning starting from the 612MB SFT checkpoint.
Saves `best_success_vera.pt` whenever task success rate improves.

**Fixes applied vs previous run:**
- All 8 rollouts accumulated before ONE gradient update (was 4 separate updates)
- KL warmup: zero KL penalty for first 20 epochs so policy adapts to PyBullet first
- Cosine LR schedule: decays from 1e-5 → 1e-6 over 200 epochs
- Best checkpoint saved by **success rate**, not mean return
- Language-Table env wrapper properly wired to rl_trainer_vera.py

In [ ]:
# ── Cell 1: GPU check ─────────────────────────────────────────────────────────
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# ── Cell 3: Install dependencies ─────────────────────────────────────────────
import subprocess, sys

def pip(*args, verbose=False):
    flags = [] if verbose else ['-q']
    subprocess.run([sys.executable, '-m', 'pip', 'install'] + flags + list(args), check=True)

# pybullet first — needed by MinimalPushEnv fallback AND by language_table itself
pip('pybullet', 'dm-env', 'absl-py', 'Pillow')
print('pybullet + dm-env: installed')

# language-table: DeepMind package for the real LT sim.
# If this fails, sim_env.py automatically falls back to MinimalPushEnv (pure pybullet).
try:
    pip('language-table')
    import language_table
    print('language-table: installed from PyPI')
except Exception:
    try:
        # GitHub zip — install deps explicitly first so the build doesn't fail on missing headers
        pip('tensorflow-cpu', 'tf-agents')   # LT hard deps; tf is already in Colab
        pip('https://github.com/google-deepmind/language_table/archive/refs/heads/main.zip',
            verbose=True)                     # verbose so we can see any build errors
        import language_table
        print('language-table: installed from GitHub zip')
    except Exception as e:
        print(f'NOTE: language-table install failed ({type(e).__name__}: {e})')
        print('      sim_env.py will auto-fallback to MinimalPushEnv (pure PyBullet) — training will still run.')

# CLIP
try:
    import clip
    print('clip: already available')
except ImportError:
    try:
        pip('https://github.com/openai/CLIP/archive/refs/heads/main.zip')
        print('clip: installed from GitHub zip')
    except Exception as e:
        print(f'WARNING: CLIP install failed: {e}')

pip('torchvision', 'pyyaml')
print('All dependencies done.')


In [ ]:

# ── Cell 4: Copy VERA code from Drive to Colab runtime ───────────────────────
# SET THIS to wherever you uploaded the TERM/Latest folder on your Drive.
# From your last run: /content/drive/MyDrive/VERA_latest/TERM/Latest
VERA_DRIVE_PATH = '/content/drive/MyDrive/VERA_latest/TERM/Latest'  # <-- CHANGE IF NEEDED

import os, shutil
VERA_RUNTIME = '/content/VERA'
if os.path.exists(VERA_RUNTIME):
    shutil.rmtree(VERA_RUNTIME)
shutil.copytree(VERA_DRIVE_PATH, VERA_RUNTIME)
print(f'Copied VERA code to {VERA_RUNTIME}')
print('Contents:', os.listdir(VERA_RUNTIME))


In [ ]:
# ── Cell 5: Set Python path ───────────────────────────────────────────────────
import sys
if VERA_RUNTIME not in sys.path:
    sys.path.insert(0, VERA_RUNTIME)
os.chdir(VERA_RUNTIME)
print('Working dir:', os.getcwd())
print('sys.path[0]:', sys.path[0])

In [ ]:

# ── Cell 6: Locate SFT checkpoint ────────────────────────────────────────────
# Derived automatically from VERA_DRIVE_PATH set in Cell 4.
# The checkpoint lives inside the results/ tree of your Drive copy.

SFT_CKPT = os.path.join(
    VERA_DRIVE_PATH,
    'results/calvin_comparisons_ltdev/01_vera_full/seed42/best_sft_vera.pt'
)
print(f'Looking for SFT checkpoint at:\n  {SFT_CKPT}')

if not os.path.exists(SFT_CKPT):
    # Show what IS in results/ to help diagnose path issues
    results_dir = os.path.join(VERA_DRIVE_PATH, 'results')
    if os.path.exists(results_dir):
        print('\nResults directory exists. Contents (up to 3 levels):')
        for root, dirs, files in os.walk(results_dir):
            depth = root.replace(results_dir, '').count(os.sep)
            if depth > 2:
                continue
            indent = '  ' * depth
            print(f'{indent}{os.path.basename(root)}/')
            for f in files:
                print(f'{indent}  {f}')
    else:
        print(f'\nResults directory not found: {results_dir}')
    raise AssertionError(f'SFT checkpoint not found: {SFT_CKPT}\n'
                         'Check the tree above and update VERA_DRIVE_PATH or the path below.')

size_mb = os.path.getsize(SFT_CKPT) / 1e6
print(f'\nSFT checkpoint found: {size_mb:.0f} MB')


In [ ]:
# ── Cell 7: Patch config so output_dir points to Drive ───────────────────────
import yaml
from pathlib import Path

CFG_PATH = f'{VERA_RUNTIME}/configs/config.yaml'
with open(CFG_PATH) as f:
    cfg = yaml.safe_load(f)

# Point output_dir to the Drive folder that contains best_sft_vera.pt
SFT_DIR = str(Path(SFT_CKPT).parent)
cfg['training']['output_dir'] = SFT_DIR

# Confirm env is set to language_table
cfg['env']['env_id'] = 'language_table'

# Write patched config back
PATCHED_CFG = f'{VERA_RUNTIME}/configs/config_colab.yaml'
with open(PATCHED_CFG, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config patched:')
print(f'  output_dir : {cfg["training"]["output_dir"]}')
print(f'  env_id     : {cfg["env"]["env_id"]}')
print(f'  num_rollouts: {cfg["rl"]["num_rollouts"]}')
print(f'  kl_coef    : {cfg["rl"]["kl_coef"]}')
print(f'  kl_warmup  : {cfg["rl"]["kl_warmup_epochs"]} epochs')

In [ ]:

# ── Cell 8: Smoke-test environment ────────────────────────────────────────────
# Uses make_env() which auto-falls-back to MinimalPushEnv if language_table is not installed.
from envs.sim_env import make_env
import numpy as np

print('Testing environment via make_env()...')
env = make_env(cfg)   # LanguageTableEnv if LT installed, else MinimalPushEnv
obs = env.reset()
print(f'  reset() frame shape : {obs["frame"].shape}')
print(f'  instruction         : "{obs["instruction"][:80]}"')

obs2, r, done, info = env.step(0)   # action 0 = push right
print(f'  step(0) reward={r:.4f}  done={done}  info={info}')
env.close()
print('Environment OK.')


In [ ]:
# ── Cell 9: Smoke-test model loads from SFT checkpoint ───────────────────────
import torch
from evaluation.evaluate_vera import build_vera_from_cfg, load_checkpoint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = build_vera_from_cfg(cfg, device)
load_checkpoint(model, SFT_CKPT, device)
print(f'Model loaded. Trainable params: {model.num_trainable_params():,}')
print(model.param_summary())

In [ ]:
# ── Cell 10: Run TERM-RL training ────────────────────────────────────────────
# This is the main training cell. Expected runtime: ~6-12h on A100 for 200 epochs.
# Results are saved to Drive every 5 epochs. Reconnect and rerun from Cell 11 if
# Colab disconnects.
#
# Watch for lines like:
#   RL Epoch  23 | steps   9200 | return 0.1832 | success 12.5% | ...
#   ★ best-success checkpoint saved (success=12.5% @ epoch 23, 9200 steps)

from training.rl_trainer_vera import rl_train

with open(PATCHED_CFG) as f:
    cfg_run = yaml.safe_load(f)

rl_train(cfg_run)

In [ ]:
# ── Cell 11: Resume after disconnect ─────────────────────────────────────────
# If Colab disconnected, run Cells 1-7 first to reinstall and remount Drive,
# then run this cell. It resumes from the last saved epoch checkpoint.
#
# The trainer always starts from best_sft_vera.pt — the RL epoch checkpoints
# are separate. To resume from a specific RL epoch, load it manually:

RESUME_FROM = None   # e.g. f'{SFT_DIR}/rl/rl_vera_epoch0040.pt'
                     # None = start fresh RL from SFT checkpoint

if RESUME_FROM is not None:
    from training.rl_trainer_vera import rl_train
    import yaml
    with open(PATCHED_CFG) as f:
        cfg_run = yaml.safe_load(f)
    # Point the trainer at the RL checkpoint to resume
    cfg_run['training']['output_dir'] = str(Path(RESUME_FROM).parent.parent)
    rl_train(cfg_run)
else:
    print('Set RESUME_FROM to a checkpoint path to resume, or re-run Cell 10 to start fresh.')

In [ ]:
# ── Cell 12: Plot results ─────────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt

LOG_PATH = f'{SFT_DIR}/rl/rl_vera_log.json'
assert os.path.exists(LOG_PATH), f'Log not found: {LOG_PATH}'

with open(LOG_PATH) as f:
    log = json.load(f)

epochs   = [r['epoch']        for r in log]
success  = [r['success_rate'] * 100 for r in log]
ret      = [r['mean_return']  for r in log]
entropy  = [r.get('entropy', 0) for r in log]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, success, 'b-o', markersize=3)
axes[0].set_title('Task Success Rate (%)')
axes[0].set_xlabel('Epoch')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, ret, 'g-o', markersize=3)
axes[1].set_title('Mean Episode Return')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, entropy, 'r-o', markersize=3)
axes[2].set_title('Policy Entropy')
axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SFT_DIR}/rl/training_curves.png', dpi=150)
plt.show()

best_success_epoch = max(log, key=lambda r: r['success_rate'])
print(f'\nBest success: {best_success_epoch["success_rate"]*100:.1f}% at epoch {best_success_epoch["epoch"]}')
print(f'Total epochs: {len(log)}')

In [ ]:
# ── Cell 13: Greedy eval on best-success checkpoint ──────────────────────────
# Run 50 greedy episodes on the best-success checkpoint to get the paper number.

from evaluation.evaluate_vera import build_vera_from_cfg, load_checkpoint, evaluate_once

BEST_SUCCESS_CKPT = f'{SFT_DIR}/rl/best_success_vera.pt'
assert os.path.exists(BEST_SUCCESS_CKPT), 'No best_success_vera.pt yet — training may not have found any successes'

model_eval = build_vera_from_cfg(cfg, device)
load_checkpoint(model_eval, BEST_SUCCESS_CKPT, device)

results = evaluate_once(model_eval, cfg, num_episodes=50, deterministic=True, seed=42)
print('\n── Greedy Evaluation (50 episodes) ──────────────────────────')
print(f'  Task success rate : {results["success_rate"]*100:.1f}%')
print(f'  Mean return       : {results["mean_return"]:.4f} ± {results["std_return"]:.4f}')
print(f'  Mean ep length    : {results["mean_length"]:.1f} steps')
print(f'  Policy entropy    : {results["mean_entropy"]:.3f}')
print(f'\n  Action distribution: {results["action_dist"]}')